In [ ]:
import cracknuts as cn
from cracknuts.cracker import serial

In [ ]:
o1 = cn.cracker_o1('192.168.0.23')
o1.connect(force_update_bin=True, force_write_default_config=True)

In [ ]:
o1.get_hardware_model()

In [ ]:
# 配置与读取 数字接口。以下接口的pin_id 参数为板子上的丝印名称
o1.digital_pin_mode('a', 1) # 配置/为输入模式
# pause ,此时外部输入高电平，下面应该打印 1
print(o1.digital_read('a')) # 读取
# o1.digital_pin_mode('a', 0) # 配置为输出模式
# o1.digital_write('a', 1) # 设置高电平, 然后测量改pin脚的电压应为 3.3

In [ ]:
# led显示
# 显示文字
o1.set_led_text('Hello World!', y = 9, x = 3)
# 显示图片
o1.set_led_image(r'D:\work\10.document\01.CrackNuts\网站相关内容\logo_2.png')

In [ ]:
# switch
o1.get_switch_status_pl()
o1.get_switch_status_ps() # 此处未开发，返回的是pl的数据

In [ ]:
# xadc
print(o1.get_voltage_a0())
print(o1.get_voltage_a1())

In [ ]:
# dac sin
o1.set_waveform_sine(frequency='8m', vpp=1.5, offset=2, phase=0.3)

In [ ]:
# dac square
o1.set_waveform_square(frequency='100k', vpp=1.5, duty=0.8, offset=2)

In [ ]:
# dac triangle
o1.set_waveform_triangle(frequency='2m', vpp=1.6, offset=0.8)

In [ ]:
# dac sawtooth
o1.set_waveform_sawtooth(frequency='1m', vpp=2, offset=1, slope=0)

In [ ]:
# 基础波形设置函数
o1.set_waveform_standard(waveform='sine', frequency=1_000_000, vpp=2.5)

In [ ]:
# dac dc 波形
o1.set_waveform_dc(voltage=3)

In [ ]:
# dac 任意波形设置
# 以下设置一个周期为 40*6.25ns的方波 频率为4mhz
o1.set_waveform_arbitrary(
    wave=[
    3.2, 3.2, 3.2, 3.2, 3.2, 3.2, 3.2, 3.2, 3.2, 3.2, 3.2, 3.2, 3.2, 3.2, 3.2, 3.2, 3.2, 3.2, 3.2, 3.2,
    0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0
])

In [ ]:
# 文件加载波形
o1.set_waveform_from_file(file_path=r'C:\Users\Dingzb\Desktop\wav.txt')

In [ ]:
# adc
# 1. cha 和 chab 分别接入信号发生器的正弦信号进行，然后使用 面板的 示波器功能（运行模式）采集波形
# 2. 配置波形发生器的波形，并接入到cha和chb 可实现自监测
acq = cn.simple_acq(o1)
cn.show_panel(acq)

In [ ]:
## O1 EDU-EV_V1 板

In [ ]:
# pwm 风扇
# 配置gpio29引脚，10khz的方波，占空比0.5
o1.set_pwm_gp29(10_000, 0.5)
# 配置gpio28引脚，10khz的方波，占空比0.8
# o1.set_pwm_gp28(10_000, 0.8)

In [ ]:
# pwm 电机
# 配置gpio29引脚，10khz的方波，占空比0.5
# o1.set_pwm_gp29(1600, 0.5) # 比较平稳的最大值
# o1.set_pwm_gp29(600, 0.5) # 比较平稳的最小值
o1.set_pwm_gp29(600, 0.0) # 关闭

In [ ]:
# LED 数字

import time


GPIO_A = 'GP5'
GPIO_B = 'GP4'
GPIO_C = 'GP7'
GPIO_D = 'GP6'
GPIO_E = 'GP22'
GPIO_F = 'GP21'
GPIO_G = 'GP24'
GPIO_DP = 'GP23'

o1.digital_pin_mode(GPIO_A, 'OUTPUT')
o1.digital_pin_mode(GPIO_B, 'OUTPUT')
o1.digital_pin_mode(GPIO_C, 'OUTPUT')
o1.digital_pin_mode(GPIO_D, 'OUTPUT')
o1.digital_pin_mode(GPIO_E, 'OUTPUT')
o1.digital_pin_mode(GPIO_F, 'OUTPUT')
o1.digital_pin_mode(GPIO_G, 'OUTPUT')
o1.digital_pin_mode(GPIO_DP, 'OUTPUT')

# 低电平亮
digit_map = {
    0: 0b0000001,
    1: 0b1001111,
    2: 0b0010010,
    3: 0b0000110,
    4: 0b1001100,
    5: 0b0100100,
    6: 0b0100000,
    7: 0b0001111,
    8: 0b0000000,
    9: 0b0000100,
    -1: 0b11111111
}

pins = [
    GPIO_A,
    GPIO_B,
    GPIO_C,
    GPIO_D,
    GPIO_E,
    GPIO_F,
    GPIO_G,
]

def display_digit(n):
    value = digit_map[n]

    for i, pin in enumerate(pins):
        bit = (value >> (6 - i)) & 1
        o1.digital_write(pin, bit)

    # 小数点默认关闭（高电平）
    o1.digital_write(GPIO_DP, 1)

for i in range(10):
    display_digit(i)
    time.sleep(1)

# 关闭数字显示
display_digit(-1)

# 闪烁
for i in range(10):
    o1.digital_write(GPIO_DP, i%2)
    time.sleep(0.1)

In [ ]:
# 红绿灯

import time

# 东
E_RED = 'GP26'
E_GREEN = 'GP27'
E_YELLOW = 'GP28'

# 南
S_GREEN = 'GP24'
S_YELLOW = 'GP25'
S_RED = 'GP23'

# 西
W_RED = 'GP7'
W_GREEN = 'GP21'
W_YELLOW = 'GP22'

# 北
N_RED = 'GP4'
N_YELLOW = 'GP6'
N_GREEN = 'GP5'


# 设置 GPIO 模式
pins = [
    # E_RED, E_YELLOW, E_GREEN,
     E_RED, E_GREEN,
    S_RED, S_GREEN, S_YELLOW,
    W_RED, W_YELLOW, W_GREEN,
    N_RED, N_YELLOW, N_GREEN
]

for p in pins:
    o1.digital_pin_mode(p, 'OUTPUT')


def all_off():
    for p in pins:
        o1.digital_write(p, 1)


all_off()
for _ in range(5):
    for p in pins:
        o1.digital_write(p, 0)
        time.sleep(0.5)
        o1.digital_write(p, 1)
all_off()

In [ ]:
# 流水灯
import time

# LED 引脚 (GPIOXX -> GPXX)
LED_PINS = [
    'GP4',   # D8
    'GP5',   # D9
    'GP6',   # D10
    'GP7',   # D11
    'GP21',  # D12
    'GP22',  # D13
    'GP23',  # D14
    'GP24',  # D15
]

# 初始化 GPIO
for p in LED_PINS:
    o1.digital_pin_mode(p, 'OUTPUT')
    o1.digital_write(p, 1)   # 默认灭 (1=灭)


def all_off():
    for p in LED_PINS:
        o1.digital_write(p, 1)


def light(index):
    all_off()
    o1.digital_write(LED_PINS[index], 0)  # 0 = 亮


# 循环运行
for _ in range(2):

    # 左 -> 右
    for i in range(8):
        light(i)
        time.sleep(0.15)

    # 右 -> 左
    for i in range(7, -1, -1):
        light(i)
        time.sleep(0.15)

    # 两端 -> 中间
    for i in range(4):
        all_off()
        o1.digital_write(LED_PINS[i], 0)
        o1.digital_write(LED_PINS[7-i], 0)
        time.sleep(0.2)

    # 中间 -> 两端
    for i in range(3, -1, -1):
        all_off()
        o1.digital_write(LED_PINS[i], 0)
        o1.digital_write(LED_PINS[7-i], 0)
        time.sleep(0.2)

all_off()

In [ ]:
# LED 
import time

LED_GPIO = 'GP4'

o1.digital_pin_mode(LED_GPIO, 'OUTPUT')


def on():
    o1.digital_write(LED_GPIO, 1)  # 亮


def off():
    o1.digital_write(LED_GPIO, 0)  # 灭


# 整个效果循环 3 次
for _ in range(1):

    # 快速闪三下
    for _ in range(3):
        on()
        time.sleep(0.15)
        off()
        time.sleep(0.15)

    time.sleep(0.4)

    # 慢闪两下
    for _ in range(2):
        on()
        time.sleep(0.5)
        off()
        time.sleep(0.5)

    time.sleep(0.6)

    # 渐快闪烁
    delays = [0.5, 0.4, 0.3, 0.2, 0.1]

    for d in delays:
        on()
        time.sleep(d)
        off()
        time.sleep(d)

# 结束后保持熄灭
off()

In [ ]:
# 按钮

import time
from IPython.display import clear_output

# 按键 GPIO
BTN1 = 'GP3'
BTN2 = 'GP2'
BTN3 = 'GP1'
BTN4 = 'GP0'

BUTTONS = [BTN1, BTN2, BTN3, BTN4]
BUTTON_NAMES = ['BTN1', 'BTN2', 'BTN3', 'BTN4']

# 初始化输入模式
for btn in BUTTONS:
    o1.digital_pin_mode(btn, 'INPUT')

start_time = time.time()
while time.time() - start_time < 60:
    clear_output(wait=True) 
    for i, btn in enumerate(BUTTONS):
        r, val = o1.digital_read(btn)
        print(f"button {btn} value {'按下' if not val else '未按'}")
    time.sleep(0.05)  # 50ms 检测一次

In [ ]:
# 光照传感器 I2C

# 初始化
from IPython.display import clear_output
import time
from cracknuts.cracker import serial

o1.i2c_config(dev_addr=0x23, speed=serial.I2cSpeed.FAST_400K)
o1.i2c_enable()
# 初始化光线感应器
o1.i2c_transceive(tx_data='01', rx_count=1)
o1.i2c_transceive(tx_data='10', rx_count=0)

# 读取数值

for _ in range(40):
    clear_output(wait=True) 
    r, v = o1.i2c_transceive(tx_data='21', rx_count=2)
    if v and len(v) == 2:
        raw_value = (v[0] << 8) | v[1]
        print(f"{(raw_value / 1.2):.2f} lx")
    time.sleep(0.5)

In [ ]:
# 温度传感器 SPI 

from cracknuts.cracker.serial import SpiCpol, SpiCpha
import time
from IPython.display import clear_output

# device
spi = o1

# SPI init (MAX31856 requires SPI Mode 1)
spi.spi_config(
    speed=10000,
    cpol=SpiCpol.SPI_CPOL_LOW,
    cpha=SpiCpha.SPI_CPHA_HIGH,
    csn_auto=True
)
spi.spi_enable()

# registers
CR0 = 0x00
CR1 = 0x01
LTCBH = 0x0C

# SPI command masks
WRITE = 0x80
READ  = 0x7F


def write_reg(reg, val):
    addr = reg | WRITE
    tx = f"{addr:02x}{val:02x}"
    spi.spi_transmit(tx, False)


def read_regs(reg, n):
    addr = reg & READ
    tx = f"{addr:02x}"
    _, rx = spi.spi_transmit_delay_receive(tx_data=tx, delay=1000, rx_count=n, is_trigger=False)
    return rx


def init():
    # stop conversion
    write_reg(CR0, 0x00)

    # K-type thermocouple + 4 sample average
    write_reg(CR1, 0x20 | 0x03)

    # auto conversion + 50Hz filter
    write_reg(CR0, 0x80 | 0x01)


def read_temp():
    b1, b2, b3 = read_regs(LTCBH, 3)

    v = (b1 << 16) | (b2 << 8) | b3
    v >>= 5

    if v & 0x40000:
        v |= 0xFFF80000

    return v * 0.0078125


# run
init()

for i in range (40):
    clear_output(wait=True) 
    t = read_temp()
    print(f"{t:.2f} °C")
    time.sleep(0.5)

In [ ]:
# 距离传感器 UART

from cracknuts.cracker.serial import Baudrate
from IPython.display import clear_output

o1.uart_config(baudrate=Baudrate.BAUDRATE_9600)
o1.uart_io_enable()

# 设置发送周期为 500ms
o1.uart_transmit_receive(tx_data='s2-500#'.encode(), rx_count=10)

for _ in range(20):
    clear_output(wait=True) 
    o1.uart_receive_fifo_clear()
    s, r = o1.uart_receive(rx_count=10)
    v = r.decode().split('\r\r\n')
    print(f"The distance is {v[0]}")
    time.sleep(0.5)

In [ ]:
# led 矩阵，GPIO 控制

import time

DATA = "GP4"
CLK = "GP6"
LATCH = "GP5"

def led595_init():
    o1.digital_pin_mode(DATA, "OUTPUT")
    o1.digital_pin_mode(CLK, "OUTPUT")
    o1.digital_pin_mode(LATCH, "OUTPUT")
    o1.digital_write(DATA, 0)
    o1.digital_write(CLK, 0)
    o1.digital_write(LATCH, 0)

def shift_bit(bit):
    o1.digital_write(DATA, bit)
    o1.digital_write(CLK, 1)
    o1.digital_write(CLK, 0)

def shift_16bit(value):
    for i in range(15, -1, -1):
        shift_bit((value >> i) & 1)
    o1.digital_write(LATCH, 1)
    o1.digital_write(LATCH, 0)


def display_matrix(matrix, t):
    s = time.time()
    while True:
        if time.time() -s > t:
            return
        s = time.time()
        for row in range(8):
            data = matrix[row]          # 列数据
            row_sel = 1 << row          # 行选择

            # 高8位控制行，低8位控制列
            value = (row_sel << 8) | data
            _s = time.time()
            shift_16bit(value)
        time.sleep(0.001)           # 矩阵扫描延时
            
# shift_16bit(0x0100)  # 第一行

# shift_16bit(0xff00)

# 显示矩阵图(频率较低，效果为逐行扫描）
display_matrix(
    [
        0b00000000,
        0b00000000,
        0b00000000,
        0b00000000,
        0b00000000,
        0b00000000,
        0b00000000,
        0b00000000,
    ], 2
)